## 1. Mount Google Drive

In [ ]:
# Optional: Mount Google Drive only when running in Google Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; using local paths.")


## 2. Import Required Libraries

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from sklearn.utils.class_weight import compute_class_weight

## 3. Reproducibility Setup

A fixed random seed is used to make the experiment more reproducible. This helps reduce randomness in data shuffling, model initialization, and training behavior.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Seed fixed:", SEED)

## 4. Dataset and Output Paths


In [ ]:
# Set these environment variables to override the default local paths.
# Data is not included in this public repository.
DATASET_DIR = Path(os.environ.get("ROUH_DATASET_DIR", "data/embryo_images_raw"))
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"
PROJECT_DIR = Path(os.environ.get("ROUH_OUTPUT_DIR", "outputs"))
MODELS_DIR = PROJECT_DIR / "Models"
RESULTS_DIR = PROJECT_DIR / "Results"
DOCS_DIR = PROJECT_DIR / "Docs"
EXPORTS_DIR = PROJECT_DIR / "Exports"
for folder in [MODELS_DIR, RESULTS_DIR, DOCS_DIR, EXPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print("Dataset Exists:", DATASET_DIR.exists())
print("Train Exists:", TRAIN_DIR.exists())
print("Test Exists:", TEST_DIR.exists())
print("Project directories ready.")


## 5. Image Dataset Preparation

In this section, the image size, batch size, and number of epochs are defined. ImageDataGenerator is then used to prepare the training, validation, and testing data.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 30

print("Image Size:", IMG_SIZE)
print("Batch Size:", BATCH_SIZE)
print("Epochs:", EPOCHS)

## 6. Image Preprocessing and Data Augmentation

Training images are preprocessed and augmented to improve model generalization. A validation split is created from the training folder.

The test generator only applies preprocessing and does not use augmentation.

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.20,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.05,
    horizontal_flip=False
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

## 7. Create Training, Validation, and Testing Generators

The dataset is loaded from directory folders. The training generator uses 80% of the training folder, while the validation generator uses 20%. The test generator is kept separate for final evaluation.

In [ ]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    shuffle=True,
    seed=SEED
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

## 8. Dataset Split Sizes

The number of samples in the training, validation, and testing sets is displayed to document the dataset split used in the experiment.

In [ ]:
print("Training Samples:", train_generator.samples)
print("Validation Samples:", val_generator.samples)
print("Testing Samples:", test_generator.samples)

print("Class Mapping:", train_generator.class_indices)

## 9. Dataset Class Distribution

The class distribution is analyzed after creating the generators. This helps identify class imbalance between non-viable and viable embryo images.

In [ ]:
from collections import Counter

train_counts = Counter(train_generator.classes)
val_counts = Counter(val_generator.classes)
test_counts = Counter(test_generator.classes)

print("Training Class Counts:", train_counts)
print("Validation Class Counts:", val_counts)
print("Testing Class Counts:", test_counts)

## 10. Class Distribution Visualization

The training class distribution is visualized to show the imbalance between the two embryo classes.

In [ ]:
labels = ["Non-Viable (0)", "Viable (1)"]
counts = [
    train_counts[0],
    train_counts[1]
]

plt.figure(figsize=(6, 4))
plt.bar(labels, counts)
plt.title("Training Dataset Class Distribution")
plt.ylabel("Number of Images")
plt.show()

## 11. Class Imbalance Handling

Since the dataset is imbalanced, class weights are calculated. This gives more importance to the minority class during training.

In [ ]:
classes = np.unique(train_generator.classes)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_generator.classes
)

class_weights = dict(
    zip(classes, class_weights_values)
)

print("Class Weights:", class_weights)

# 12. Baseline CNN Model

A simple Convolutional Neural Network (CNN) is implemented as a baseline model.

The purpose of this model is to establish a reference performance before applying transfer learning using EfficientNetB0.

## 12.1 Build Baseline CNN Architecture

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

baseline_model = Sequential([

    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    ),

    MaxPooling2D(2,2),

    Conv2D(
        64,
        (3,3),
        activation='relu'
    ),

    MaxPooling2D(2,2),

    Conv2D(
        128,
        (3,3),
        activation='relu'
    ),

    MaxPooling2D(2,2),

    Flatten(),

    Dense(
        128,
        activation='relu'
    ),

    Dropout(0.5),

    Dense(
        1,
        activation='sigmoid'
    )
])

baseline_model.summary()

## 12.2 Compile Baseline CNN

The baseline CNN is compiled using binary crossentropy because this is a binary classification problem.

Although accuracy is included during training, additional medical evaluation metrics are also added:

- Precision
- Recall / Sensitivity
- AUC

These metrics are important because accuracy alone can be misleading in imbalanced medical datasets.

In [ ]:
baseline_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

## 12.3 Create Baseline Model Directory

A separate folder is created to save the best baseline CNN model during training.

In [ ]:
baseline_model_dir = MODELS_DIR / "baseline_cnn"

baseline_model_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Baseline model directory ready:", baseline_model_dir)

## 12.4 Training Callbacks

Three callbacks are used during training:

- EarlyStopping: stops training when validation performance stops improving.
- ReduceLROnPlateau: reduces the learning rate when improvement slows down.
- ModelCheckpoint: saves the best model during training.

Validation AUC is monitored because AUC is more informative than accuracy for imbalanced binary classification.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=str(
        baseline_model_dir / "best_baseline_model.keras"
    ),
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=1
)

## 12.5 Train Baseline CNN

The baseline CNN is trained using the training generator and evaluated on the validation generator. Class weights are applied to reduce bias toward the majority class.

In [ ]:
history_baseline = baseline_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ]
)

## 12.6 Save Baseline Training History

The training history is saved as a CSV file so it can be used later in the report, plots, and model comparison.

In [ ]:
history_baseline_df = pd.DataFrame(
    history_baseline.history
)

history_baseline_df.to_csv(
    RESULTS_DIR / "baseline_training_history.csv",
    index=False
)

history_baseline_df.head()

# 13. Baseline CNN Evaluation

The baseline CNN is evaluated on the independent test set.

Multiple metrics are reported because medical AI systems should not rely on accuracy alone.

The evaluation includes:

- Accuracy
- Precision
- Recall (Sensitivity)
- F1-score
- Specificity
- Balanced Accuracy
- ROC-AUC
- Confusion Matrix

## 13.1 Load Best Baseline Model

In [ ]:
best_baseline_model = tf.keras.models.load_model(
    baseline_model_dir / "best_baseline_model.keras"
)

print("Baseline model loaded successfully.")

## 13.2 Generate Predictions

Predictions are generated for all images in the test set.

In [ ]:
test_generator.reset()

y_prob_baseline = best_baseline_model.predict(
    test_generator,
    verbose=1
)

y_prob_baseline = y_prob_baseline.flatten()

y_pred_baseline = (
    y_prob_baseline >= 0.5
).astype(int)

y_true_baseline = test_generator.classes

## 13.3 Calculate Evaluation Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score
)

baseline_accuracy = accuracy_score(
    y_true_baseline,
    y_pred_baseline
)

baseline_precision = precision_score(
    y_true_baseline,
    y_pred_baseline,
    zero_division=0
)

baseline_recall = recall_score(
    y_true_baseline,
    y_pred_baseline,
    zero_division=0
)

baseline_f1 = f1_score(
    y_true_baseline,
    y_pred_baseline,
    zero_division=0
)

baseline_auc = roc_auc_score(
    y_true_baseline,
    y_prob_baseline
)

baseline_balanced_acc = balanced_accuracy_score(
    y_true_baseline,
    y_pred_baseline
)

print("Accuracy :", baseline_accuracy)
print("Precision:", baseline_precision)
print("Recall   :", baseline_recall)
print("F1 Score :", baseline_f1)
print("AUC      :", baseline_auc)
print("Balanced Accuracy :", baseline_balanced_acc)

## Baseline Model Interpretation

Although the baseline CNN achieved an accuracy of 85.1%, further analysis revealed that the model failed to identify any viable embryos. This is reflected by a precision, recall, and F1-score of 0.0.

The model effectively predicted only the majority class, resulting in a misleadingly high accuracy. The AUC score of approximately 0.49 and balanced accuracy of 0.50 further indicate that the model performed close to random guessing.

These findings demonstrate the limitations of a simple CNN architecture for embryo viability prediction and justify the use of transfer learning with EfficientNetB0.

## 13.5 Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

cm_baseline = confusion_matrix(
    y_true_baseline,
    y_pred_baseline
)

cm_baseline

## 13.6 Visualize Confusion Matrix

In [ ]:
plt.figure(figsize=(6,5))

plt.imshow(
    cm_baseline,
    cmap="Blues"
)

plt.title("Baseline CNN Confusion Matrix")

plt.colorbar()

plt.xticks([0,1], ["Non-Viable","Viable"])
plt.yticks([0,1], ["Non-Viable","Viable"])

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm_baseline[i,j],
            ha="center",
            va="center"
        )

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

## 13.7 Calculate Specificity

Specificity measures how well the model correctly identifies non-viable embryos.

Together, Recall (Sensitivity) and Specificity provide a more clinically meaningful evaluation.

In [ ]:
TN = cm_baseline[0,0]
FP = cm_baseline[0,1]
FN = cm_baseline[1,0]
TP = cm_baseline[1,1]

baseline_specificity = TN / (TN + FP)

print(
    "Specificity:",
    round(
        baseline_specificity,
        4
    )
)

## 13.8 Save Baseline Results

All baseline metrics are stored for later comparison with EfficientNetB0.

In [ ]:
baseline_results = pd.DataFrame({

    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Specificity",
        "Balanced Accuracy",
        "AUC"
    ],

    "Value": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1,
        baseline_specificity,
        baseline_balanced_acc,
        baseline_auc
    ]
})

baseline_results

## Baseline CNN Interpretation

The baseline CNN achieved an accuracy of 85.1%; however, further analysis revealed that the model predicted only the majority class (non-viable embryos).

The confusion matrix shows that all 143 non-viable embryos were correctly classified, while none of the 25 viable embryos were identified.

As a result:

- Precision = 0.0
- Recall = 0.0
- F1-score = 0.0
- AUC ≈ 0.49

Despite the high accuracy, the model demonstrated no ability to identify viable embryos and therefore cannot be considered clinically useful.

This highlights the limitation of relying solely on accuracy in imbalanced medical datasets and motivates the use of transfer learning with EfficientNetB0.

# 15. ResNet50 Transfer Learning

ResNet50 is evaluated as the first transfer learning model.

Transfer learning leverages feature representations learned from large-scale image datasets, allowing better performance on medical imaging tasks with limited data.

ResNet50 serves as a benchmark transfer learning architecture and will later be compared with EfficientNetB0.

## 15.1 Load Pretrained ResNet50

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D

resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

resnet_base.trainable = False

## 15.2 Build ResNet50 Classification Head

In [ ]:
x = resnet_base.output

x = GlobalAveragePooling2D()(x)

x = Dense(
    128,
    activation="relu"
)(x)

x = Dropout(0.5)(x)

output = Dense(
    1,
    activation="sigmoid"
)(x)

resnet_model = Model(
    inputs=resnet_base.input,
    outputs=output
)

resnet_model.summary()

## 15.3 Compile ResNet50

Medical evaluation metrics are included during training to provide more informative monitoring than accuracy alone.

In [ ]:
resnet_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

## 15.4 Training Configuration

In [ ]:
resnet_model_dir = MODELS_DIR / "resnet50"

resnet_model_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
early_stopping_resnet = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_resnet = ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

checkpoint_resnet = ModelCheckpoint(
    filepath=str(
        resnet_model_dir / "best_resnet50.keras"
    ),
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=1
)

## 15.5 Train ResNet50

In [ ]:
history_resnet = resnet_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[
        early_stopping_resnet,
        reduce_lr_resnet,
        checkpoint_resnet
    ]
)

## 15.6 Save Training History


In [ ]:
history_resnet_df = pd.DataFrame(
    history_resnet.history
)

history_resnet_df.to_csv(
    RESULTS_DIR / "resnet50_training_history.csv",
    index=False
)

history_resnet_df.head()

## 15.7 ResNet50 Learning Curves

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)

plt.plot(
    history_resnet.history["auc"],
    label="Train AUC"
)

plt.plot(
    history_resnet.history["val_auc"],
    label="Validation AUC"
)

plt.title("ResNet50 AUC")

plt.legend()

plt.subplot(1,2,2)

plt.plot(
    history_resnet.history["loss"],
    label="Train Loss"
)

plt.plot(
    history_resnet.history["val_loss"],
    label="Validation Loss"
)

plt.title("ResNet50 Loss")

plt.legend()

plt.show()

## 15.8 ResNet50 Evaluation

The trained ResNet50 model is evaluated on the independent test set.

Multiple evaluation metrics are reported because medical AI systems should not rely solely on accuracy, particularly when class imbalance is present.

In [ ]:
test_results_resnet = resnet_model.evaluate(
    test_generator,
    verbose=1
)

for name, value in zip(
    resnet_model.metrics_names,
    test_results_resnet
):
    print(f"{name}: {value:.4f}")

## 15.9 Generate Predictions

In [ ]:
test_generator.reset()

y_true_resnet = test_generator.classes

y_prob_resnet = resnet_model.predict(
    test_generator,
    verbose=1
).flatten()

y_pred_resnet = (
    y_prob_resnet >= 0.5
).astype(int)

## 15.10 Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true_resnet,
        y_pred_resnet,
        digits=4
    )
)

## 15.11 Confusion Matrix

In [ ]:
cm_resnet = confusion_matrix(
    y_true_resnet,
    y_pred_resnet
)

cm_resnet

## 15.12 Visualize Confusion Matrix

In [ ]:
plt.figure(figsize=(6,5))

plt.imshow(
    cm_resnet,
    cmap="Blues"
)

plt.title("ResNet50 Confusion Matrix")

plt.colorbar()

plt.xticks(
    [0,1],
    ["Non-Viable","Viable"]
)

plt.yticks(
    [0,1],
    ["Non-Viable","Viable"]
)

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm_resnet[i,j],
            ha="center",
            va="center"
        )

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

## 15.13 Calculate Additional Medical Metrics

In addition to accuracy, clinically relevant metrics are calculated:

- Precision
- Recall (Sensitivity)
- F1-score
- Specificity
- Balanced Accuracy
- ROC-AUC

In [ ]:
TN = cm_resnet[0,0]
FP = cm_resnet[0,1]
FN = cm_resnet[1,0]
TP = cm_resnet[1,1]

resnet_specificity = TN / (TN + FP)

resnet_accuracy = accuracy_score(
    y_true_resnet,
    y_pred_resnet
)

resnet_precision = precision_score(
    y_true_resnet,
    y_pred_resnet,
    zero_division=0
)

resnet_recall = recall_score(
    y_true_resnet,
    y_pred_resnet,
    zero_division=0
)

resnet_f1 = f1_score(
    y_true_resnet,
    y_pred_resnet,
    zero_division=0
)

resnet_auc = roc_auc_score(
    y_true_resnet,
    y_prob_resnet
)

resnet_balanced_acc = balanced_accuracy_score(
    y_true_resnet,
    y_pred_resnet
)

resnet_results = pd.DataFrame({

    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Specificity",
        "Balanced Accuracy",
        "AUC"
    ],

    "Value":[
        resnet_accuracy,
        resnet_precision,
        resnet_recall,
        resnet_f1,
        resnet_specificity,
        resnet_balanced_acc,
        resnet_auc
    ]
})

resnet_results

## 15.14 ResNet50 Interpretation

ResNet50 substantially outperformed the baseline CNN across all clinically relevant metrics.

While the baseline CNN failed to identify any viable embryos, ResNet50 achieved:

- Recall = 64.0%
- Precision = 41.0%
- F1-score = 50.0%
- AUC = 0.774

The confusion matrix demonstrates that ResNet50 successfully identified 16 of 25 viable embryos while maintaining reasonable performance on non-viable embryos.

These results indicate that transfer learning significantly improves embryo viability prediction compared to a simple CNN architecture.

# 16. EfficientNetB0 Transfer Learning

EfficientNetB0 was evaluated as an advanced transfer learning architecture for embryo viability prediction.

Compared to traditional CNNs and ResNet50, EfficientNetB0 provides a better balance between model size, computational efficiency, and feature extraction capability.

The architecture is expected to capture more informative embryo image features and improve classification performance.

## 16.1 Load Pretrained EfficientNetB0

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

efficientnet_base = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

efficientnet_base.trainable = False

## 16.2 Build EfficientNetB0 Classification Head

In [ ]:
x = efficientnet_base.output

x = GlobalAveragePooling2D()(x)

x = Dense(
    128,
    activation="relu"
)(x)

x = Dropout(0.5)(x)

output = Dense(
    1,
    activation="sigmoid"
)(x)

efficientnet_model = Model(
    inputs=efficientnet_base.input,
    outputs=output
)

efficientnet_model.summary()

## 16.3 Compile EfficientNetB0

The model is trained using binary crossentropy and evaluated using medically relevant metrics including Precision, Recall, and AUC.

In [ ]:
efficientnet_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

## 16.4 Create Model Directory

In [ ]:
efficientnet_model_dir = MODELS_DIR / "efficientnetb0"

efficientnet_model_dir.mkdir(
    parents=True,
    exist_ok=True
)

## 16.5 Training Configuration

Early stopping, learning rate scheduling, and model checkpointing are used to improve convergence and reduce overfitting.

In [ ]:
early_stopping_eff = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_eff = ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

checkpoint_eff = ModelCheckpoint(
    filepath=str(
        efficientnet_model_dir / "best_efficientnetb0.keras"
    ),
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=1
)

## 16.6 Train EfficientNetB0

In [ ]:
history_efficientnet = efficientnet_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[
        early_stopping_eff,
        reduce_lr_eff,
        checkpoint_eff
    ]
)

## 16.7 Save Training History

In [ ]:
history_efficientnet_df = pd.DataFrame(
    history_efficientnet.history
)

history_efficientnet_df.to_csv(
    RESULTS_DIR / "efficientnet_training_history.csv",
    index=False
)

history_efficientnet_df.head()

## 16.8 EfficientNetB0 Learning Curves

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)

plt.plot(
    history_efficientnet.history["auc"],
    label="Train AUC"
)

plt.plot(
    history_efficientnet.history["val_auc"],
    label="Validation AUC"
)

plt.title("EfficientNetB0 AUC")

plt.legend()

plt.subplot(1,2,2)

plt.plot(
    history_efficientnet.history["loss"],
    label="Train Loss"
)

plt.plot(
    history_efficientnet.history["val_loss"],
    label="Validation Loss"
)

plt.title("EfficientNetB0 Loss")

plt.legend()

plt.show()

## 16.9 EfficientNetB0 Evaluation

The trained EfficientNetB0 model is evaluated on the independent test set using multiple performance metrics.

Medical AI systems should not rely solely on accuracy. Therefore, additional metrics such as Precision, Recall, F1-score, Specificity, Balanced Accuracy, and AUC are also reported.

In [ ]:
test_results_eff = efficientnet_model.evaluate(
    test_generator,
    verbose=1
)

for name, value in zip(
    efficientnet_model.metrics_names,
    test_results_eff
):
    print(f"{name}: {value:.4f}")

## 16.10 Generate Predictions

In [ ]:
test_generator.reset()

y_true_eff = test_generator.classes

y_prob_eff = efficientnet_model.predict(
    test_generator,
    verbose=1
).flatten()

y_pred_eff = (
    y_prob_eff >= 0.5
).astype(int)

## 16.11 Classification Report

In [ ]:
print(
    classification_report(
        y_true_eff,
        y_pred_eff,
        digits=4
    )
)

## 16.12 Confusion Matrix

In [ ]:
cm_eff = confusion_matrix(
    y_true_eff,
    y_pred_eff
)

cm_eff

## 16.13 Visualize Confusion Matrix

In [ ]:
plt.figure(figsize=(6,5))

plt.imshow(
    cm_eff,
    cmap="Blues"
)

plt.title("EfficientNetB0 Confusion Matrix")

plt.colorbar()

plt.xticks(
    [0,1],
    ["Non-Viable","Viable"]
)

plt.yticks(
    [0,1],
    ["Non-Viable","Viable"]
)

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm_eff[i,j],
            ha="center",
            va="center"
        )

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

## 16.14 Calculate Additional Medical Metrics

In [ ]:
TN = cm_eff[0,0]
FP = cm_eff[0,1]
FN = cm_eff[1,0]
TP = cm_eff[1,1]

eff_specificity = TN / (TN + FP)

eff_accuracy = accuracy_score(
    y_true_eff,
    y_pred_eff
)

eff_precision = precision_score(
    y_true_eff,
    y_pred_eff,
    zero_division=0
)

eff_recall = recall_score(
    y_true_eff,
    y_pred_eff,
    zero_division=0
)

eff_f1 = f1_score(
    y_true_eff,
    y_pred_eff,
    zero_division=0
)

eff_auc = roc_auc_score(
    y_true_eff,
    y_prob_eff
)

eff_balanced_acc = balanced_accuracy_score(
    y_true_eff,
    y_pred_eff
)

eff_results = pd.DataFrame({

    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Specificity",
        "Balanced Accuracy",
        "AUC"
    ],

    "Value":[
        eff_accuracy,
        eff_precision,
        eff_recall,
        eff_f1,
        eff_specificity,
        eff_balanced_acc,
        eff_auc
    ]
})

eff_results

## 16.15 EfficientNetB0 Interpretation

EfficientNetB0 achieved the best overall performance among the evaluated image classification models.

Compared to ResNet50, EfficientNetB0 maintained the same recall (64%) while substantially improving precision and specificity. This indicates that the model was able to identify viable embryos while generating fewer false positive predictions.

The model achieved:

- Accuracy = 88.1%
- Precision = 59.3%
- Recall = 64.0%
- F1-score = 61.5%
- Balanced Accuracy = 78.2%
- AUC = 0.790

These results demonstrate that EfficientNetB0 provides a better balance between sensitivity and specificity, making it the preferred image-based model for embryo viability prediction.

# 17. EfficientNetB0 Fine-Tuning

After initial training, the top layers of EfficientNetB0 are unfrozen and further trained using a very small learning rate.

Fine-tuning allows the model to adapt ImageNet features to embryo-specific characteristics and may improve classification performance.

## 17.1 Load Best EfficientNetB0 Model

In [ ]:
best_efficientnet = tf.keras.models.load_model(
    efficientnet_model_dir / "best_efficientnetb0.keras"
)

print("Best EfficientNetB0 loaded.")

## 17.2 Unfreeze Top Layers

In [ ]:
best_efficientnet.trainable = True

for layer in best_efficientnet.layers[:-20]:
    layer.trainable = False

print(
    "Trainable layers:",
    sum([layer.trainable for layer in best_efficientnet.layers])
)

## 17.3 Recompile for Fine-Tuning

In [ ]:
best_efficientnet.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

## 17.4 Fine-Tuning Training

In [ ]:
history_finetune = best_efficientnet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=3,
            restore_best_weights=True
        )
    ]
)

## 17.5 Save Fine-Tuned Model

In [ ]:
best_efficientnet.save(
    efficientnet_model_dir /
    "best_efficientnet_finetuned.keras"
)

## 17.6 Fine-Tuning Learning Curves

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)

plt.plot(
    history_finetune.history["val_auc"],
    label="Validation AUC"
)

plt.legend()

plt.title(
    "Fine-Tuning Validation AUC"
)

plt.subplot(1,2,2)

plt.plot(
    history_finetune.history["val_loss"],
    label="Validation Loss"
)

plt.legend()

plt.title(
    "Fine-Tuning Validation Loss"
)

plt.show()

## 17.7 Fine-Tuned EfficientNetB0 Evaluation

The fine-tuned model is evaluated on the independent test set using the same evaluation framework applied to the previous models.

This ensures a fair comparison between:
- Baseline CNN
- ResNet50
- EfficientNetB0
- Fine-Tuned EfficientNetB0

In [ ]:
test_results_finetune = best_efficientnet.evaluate(
    test_generator,
    verbose=1
)

for name, value in zip(
    best_efficientnet.metrics_names,
    test_results_finetune
):
    print(f"{name}: {value:.4f}")

## 17.8 Generate Predictions

In [ ]:
test_generator.reset()

y_true_ft = test_generator.classes

y_prob_ft = best_efficientnet.predict(
    test_generator,
    verbose=1
).flatten()

y_pred_ft = (
    y_prob_ft >= 0.5
).astype(int)

## 17.9 Classification Report

In [ ]:
print(
    classification_report(
        y_true_ft,
        y_pred_ft,
        digits=4
    )
)

## 17.10 Confusion Matrix

In [ ]:
cm_ft = confusion_matrix(
    y_true_ft,
    y_pred_ft
)

cm_ft

## 17.11 Calculate Additional Medical Metrics

In [ ]:
TN = cm_ft[0,0]
FP = cm_ft[0,1]
FN = cm_ft[1,0]
TP = cm_ft[1,1]

ft_specificity = TN / (TN + FP)

ft_accuracy = accuracy_score(
    y_true_ft,
    y_pred_ft
)

ft_precision = precision_score(
    y_true_ft,
    y_pred_ft,
    zero_division=0
)

ft_recall = recall_score(
    y_true_ft,
    y_pred_ft,
    zero_division=0
)

ft_f1 = f1_score(
    y_true_ft,
    y_pred_ft,
    zero_division=0
)

ft_auc = roc_auc_score(
    y_true_ft,
    y_prob_ft
)

ft_balanced_acc = balanced_accuracy_score(
    y_true_ft,
    y_pred_ft
)

finetune_results = pd.DataFrame({

    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Specificity",
        "Balanced Accuracy",
        "AUC"
    ],

    "Value":[
        ft_accuracy,
        ft_precision,
        ft_recall,
        ft_f1,
        ft_specificity,
        ft_balanced_acc,
        ft_auc
    ]
})

finetune_results

## 17.12 Fine-Tuning Interpretation

Fine-tuning produced only a marginal improvement in AUC while all other evaluation metrics remained unchanged.

The confusion matrix and classification metrics indicate that the model's predictive behavior was effectively identical before and after fine-tuning.

Therefore, fine-tuning did not provide a substantial performance benefit for this dataset.

For simplicity and computational efficiency, the original EfficientNetB0 model was retained as the primary image classification model.

# 18. Threshold Optimization

The default classification threshold of 0.50 may not be optimal for embryo viability prediction.

Alternative thresholds are evaluated to improve sensitivity (Recall) and identify a larger proportion of viable embryos.

In [ ]:
thresholds = [
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60
]

threshold_results = []

for threshold in thresholds:

    preds = (
        y_prob_eff >= threshold
    ).astype(int)

    threshold_results.append({

        "Threshold": threshold,

        "Precision": precision_score(
            y_true_eff,
            preds,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true_eff,
            preds,
            zero_division=0
        ),

        "F1": f1_score(
            y_true_eff,
            preds,
            zero_division=0
        ),

        "Balanced Accuracy":
        balanced_accuracy_score(
            y_true_eff,
            preds
        )
    })

threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df

## 18.1 Threshold Optimization Interpretation

Multiple classification thresholds were evaluated to investigate whether embryo viability prediction could be improved.

Although lower thresholds slightly increased recall, they also produced a noticeable reduction in precision.

The default threshold of 0.50 achieved the highest F1-score and balanced accuracy while maintaining strong recall and specificity.

Therefore, a threshold of 0.50 was retained for the final image classification model.

# 19. Model Comparison

The performance of all evaluated image classification models is compared using medically relevant metrics.

The comparison includes:

- Baseline CNN
- ResNet50
- EfficientNetB0

The goal is to identify the most suitable model for embryo viability prediction.

In [ ]:
comparison_df = pd.DataFrame({

    "Model":[
        "Baseline CNN",
        "ResNet50",
        "EfficientNetB0"
    ],

    "Accuracy":[
        baseline_accuracy,
        resnet_accuracy,
        eff_accuracy
    ],

    "Precision":[
        baseline_precision,
        resnet_precision,
        eff_precision
    ],

    "Recall":[
        baseline_recall,
        resnet_recall,
        eff_recall
    ],

    "F1":[
        baseline_f1,
        resnet_f1,
        eff_f1
    ],

    "Specificity":[
        baseline_specificity,
        resnet_specificity,
        eff_specificity
    ],

    "Balanced Accuracy":[
        baseline_balanced_acc,
        resnet_balanced_acc,
        eff_balanced_acc
    ],

    "AUC":[
        baseline_auc,
        resnet_auc,
        eff_auc
    ]
})

comparison_df

## 19.1 Visual Comparison of Models

In [ ]:
comparison_df.set_index(
    "Model"
)[["F1","Recall","AUC"]].plot(
    kind="bar",
    figsize=(10,5)
)

plt.title(
    "Comparison of Image Classification Models"
)

plt.ylabel(
    "Score"
)

plt.xticks(rotation=0)

plt.show()

## 19.2 Model Comparison Interpretation

The Baseline CNN achieved the highest accuracy but failed to identify viable embryos, resulting in poor clinical utility.

ResNet50 significantly improved embryo viability detection and demonstrated the effectiveness of transfer learning.

EfficientNetB0 achieved the strongest overall performance, providing the highest precision, F1-score, balanced accuracy, and AUC while maintaining the same recall as ResNet50.

Therefore, EfficientNetB0 was selected as the final image-based embryo classification model.

# 20. Final Image Model Selection

Based on the experimental results, EfficientNetB0 was selected as the final image classification model.

Reasons for selection:

- Highest AUC
- Highest F1-score
- Highest Precision
- Highest Balanced Accuracy
- Strong Recall
- Best overall clinical performance

This model is used in the ROUH system as the image-based prediction component.

In [ ]:
final_image_results = pd.DataFrame({

    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Specificity",
        "Balanced Accuracy",
        "AUC"
    ],

    "EfficientNetB0":[
        eff_accuracy,
        eff_precision,
        eff_recall,
        eff_f1,
        eff_specificity,
        eff_balanced_acc,
        eff_auc
    ]
})

final_image_results

# 21. Grad-CAM Explainability

To improve transparency and interpretability, Grad-CAM (Gradient-weighted Class Activation Mapping) is applied to the final EfficientNetB0 model.

Grad-CAM highlights image regions that contribute most strongly to the model's prediction.

This helps verify that the model focuses on biologically relevant embryo structures rather than irrelevant image artifacts.

## 21.1 Select Final EfficientNetB0 Model

In [ ]:
final_model = efficientnet_model

print("Final image model loaded.")

In [ ]:
final_model = tf.keras.models.load_model(
    MODELS_DIR / "efficientnetb0" / "best_efficientnetb0.keras"
)

In [ ]:
print(final_model)

## 21.2 Identify Final Convolutional Layer

In [ ]:
for layer in reversed(final_model.layers):
    if len(layer.output.shape) == 4:
        print(layer.name)
        break

## 21.3 Grad-CAM Function

In [ ]:
def make_gradcam_heatmap(
    img_array,
    model,
    last_conv_layer_name
):

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(img_array)

        class_channel = predictions[:, 0]

    grads = tape.gradient(
        class_channel,
        conv_outputs
    )

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0, 1, 2)
    )

    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(
        conv_outputs * pooled_grads,
        axis=-1
    )

    heatmap = tf.maximum(
        heatmap,
        0
    )

    max_heat = tf.reduce_max(heatmap)

    if max_heat > 0:
        heatmap = heatmap / max_heat

    return heatmap.numpy()

## 21.4 Select Example Embryo Image

In [ ]:
# Provide an authorized example image via ROUH_SAMPLE_IMAGE or select
# the first available image from the test split.
sample_file = os.environ.get("ROUH_SAMPLE_IMAGE")
if sample_file:
    sample_path = sample_file
else:
    candidates = sorted(TEST_DIR.rglob("*.jpg")) + sorted(TEST_DIR.rglob("*.png"))
    if not candidates:
        raise FileNotFoundError("Set ROUH_SAMPLE_IMAGE to an authorized example image.")
    sample_path = str(candidates[0])
print("Example image selected.")


## 21.5 Load and Prepare Image

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input

img = Image.open(
    sample_path
).convert("RGB")

img = img.resize(
    (224,224)
)

img_array = np.array(
    img
).astype(np.float32)

display_img = img_array.astype(np.uint8).copy()

img_array = preprocess_input(
    img_array
)

img_array = np.expand_dims(
    img_array,
    axis=0
)

In [ ]:
prediction = final_model.predict(
    img_array,
    verbose=0
)

print(prediction)

## 21.6 Generate Grad-CAM Heatmap

In [ ]:
heatmap = make_gradcam_heatmap(
    img_array,
    final_model,
    "top_activation"
)

plt.figure(figsize=(6,6))

plt.imshow(
    heatmap,
    cmap="jet"
)

plt.colorbar()

plt.title(
    "Grad-CAM Heatmap"
)

plt.show()

## 21.7 Overlay Grad-CAM on Embryo Image

In [ ]:
import cv2

heatmap_resized = cv2.resize(
    heatmap,
    (
        display_img.shape[1],
        display_img.shape[0]
    )
)

heatmap_uint8 = np.uint8(
    255 * heatmap_resized
)

heatmap_color = cv2.applyColorMap(
    heatmap_uint8,
    cv2.COLORMAP_JET
)

superimposed_img = cv2.addWeighted(
    display_img,
    0.6,
    heatmap_color,
    0.4,
    0
)

plt.figure(figsize=(8,8))

plt.imshow(
    cv2.cvtColor(
        superimposed_img,
        cv2.COLOR_BGR2RGB
    )
)

plt.axis("off")

plt.title(
    "Grad-CAM Overlay"
)

plt.show()

## 21.8 Save Grad-CAM Results

In [ ]:
plt.imsave(
    RESULTS_DIR / "gradcam_heatmap.png",
    heatmap,
    cmap="jet"
)

cv2.imwrite(
    str(
        RESULTS_DIR / "gradcam_overlay.png"
    ),
    superimposed_img
)

print("Grad-CAM results saved.")